# Baseline: Gemma 3 clinical SOAP summaries (before fine-tuning)

This notebook generates summaries with the **original** `google/gemma-3-270m-it`, then uses DeepEval G-Eval to score factual faithfulness and coverage on N_CASES selected (default 10) validation examples. It does not train a model. Scores are approximate LLM judgments, not clinical validation.

**Before starting:** Open in Colab, select a GPU runtime, accept the [Gemma license](https://huggingface.co/google/gemma-3-270m-it), and create a Hugging Face access token. A Gemini API key with appropriate quota or billing is needed only for the separate judging step. Use Colab Secrets or environment variables, never paste keys into this notebook. For Colab Secrets, add `HF_TOKEN` and `GOOGLE_API_KEY` and enable notebook access.

**Cost and data:** Model/dataset downloads and GPU runtime may incur provider charges. The optional judge sends clinical notes, references and generated summaries to Google Gemini and may incur API charges: 10 cases × 2 metrics means at least 20 judge requests, with possible retries. Cost depends on token counts and current [Gemini API pricing](https://ai.google.dev/gemini-api/docs/pricing); inspect the selection and pricing first. Dataset sources are described as de-identified, but treat all saved and transmitted text as sensitive. Avoid Confident AI login for this demo. DeepEval telemetry is disabled below.

**Persistence:** Colab's local disk is temporary. Download the `baseline_evaluation_diagnostic_10` folder or set `OUTPUT_DIR` to a mounted Drive folder. Keep `selection.json` unchanged for a later fine-tuned comparison; it stores the exact inputs and references, indices, hashes and dataset revision. The saved JSON files contain clinical text.


In [1]:
%pip install -q --no-warn-conflicts "transformers>=4.55,<5" "datasets>=3,<5" accelerate "deepeval>=4.3.3,<5" google-genai pandas

In [2]:
import os, json, hashlib, random, importlib.metadata
from pathlib import Path
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"
os.environ["DEEPEVAL_EVAL_MODE"] = "llm"
# Colab Secrets are optional if environment variables already exist.
try:
    from google.colab import userdata
    for key in ("HF_TOKEN", "GOOGLE_API_KEY"):
        if not os.environ.get(key):
            try:
                value = userdata.get(key)
                if value:
                    os.environ[key] = value
            except Exception:
                pass
except ImportError:
    pass

DATASET_ID = "AmareshHebbar/clinical-summarizer-sft"
MODEL_ID = "google/gemma-3-270m-it"
SEED = 42
N_CASES = 10  # Change this to control the inference sample size.
MAX_NEW_TOKENS = 2048
assert N_CASES > 0 and MAX_NEW_TOKENS > 0
JUDGE_MODEL = "gemini-3.1-flash-lite"  # Gemini API judge, separate from Gemma
OUTPUT_DIR = Path(f"baseline_evaluation_diagnostic_{N_CASES}")
PREVIOUS_SELECTION = Path("baseline_evaluation/selection.json")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("CUDA available:", torch.cuda.is_available())
print("Output directory:", OUTPUT_DIR.resolve())


CUDA available: True
Output directory: /content/baseline_evaluation_diagnostic_10


In [6]:
import os
from getpass import getpass
from huggingface_hub import HfApi

os.environ["HF_TOKEN"] = getpass("Hugging Face token: ").strip()

account = HfApi().whoami(token=os.environ["HF_TOKEN"])
print("Authenticated as:", account["name"])

Authenticated as: Atharva138yg


## 1. Freeze the validation sample

The first run reuses the first N_CASES from the previous selection when available; otherwise it samples without replacement using seed 42, resolves the dataset's current commit, and saves the full selected examples. Subsequent runs load the existing selection rather than resampling. Save this file for your future comparison. Delete or rename the output folder only if you deliberately want a new experiment.


In [7]:
selection_path = OUTPUT_DIR / "selection.json"
if selection_path.exists():
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    assert selection["dataset"] == DATASET_ID and len(selection["cases"]) == N_CASES
    print("Reusing saved selection")
else:
    if PREVIOUS_SELECTION.exists():
        previous = json.loads(PREVIOUS_SELECTION.read_text(encoding="utf-8"))
        assert previous["dataset"] == DATASET_ID
        revision = previous["revision"]
    else:
        previous = None
        revision = HfApi().dataset_info(DATASET_ID, token=os.environ.get("HF_TOKEN")).sha
    if previous is not None and N_CASES <= len(previous["cases"]):
        cases = previous["cases"][:N_CASES]
        print("Using the first", N_CASES, "cases from the previous selection")
    else:
        validation = load_dataset(DATASET_ID, revision=revision, split="validation")
        indices = random.Random(SEED).sample(range(len(validation)), N_CASES)
        cases = []
        for index in indices:
            messages = validation[index]["messages"]
            by_role = {m["role"]: m["content"] for m in messages}
            assert all(by_role.get(role) for role in ("system", "user", "assistant")), index
            raw = json.dumps(messages, ensure_ascii=False, sort_keys=True).encode("utf-8")
            cases.append({"index": index, "sha256": hashlib.sha256(raw).hexdigest(),
                          "system": by_role["system"], "note": by_role["user"],
                          "reference": by_role["assistant"]})
    selection = {"dataset": DATASET_ID, "revision": revision, "split": "validation",
                 "seed": SEED, "cases": cases}
    selection_path.write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding="utf-8")
print("Dataset revision:", selection["revision"])
print("Selected validation row indices:", [c["index"] for c in selection["cases"]])
print("Saved:", selection_path)


Reusing saved selection
Dataset revision: 4ca20e0a38cc9235045c973b5503d49952fd1fac
Selected validation row indices: [2619, 456, 102, 1126, 1003, 914, 571, 419, 2771, 2233]
Saved: baseline_evaluation_diagnostic_10/selection.json


## 2. Generate baseline summaries

Gemma receives each dataset system instruction and note, combined into a user message, as in the training notebook. Greedy decoding makes generation repeatable for a fixed software/hardware setup. The reference summary is never sent to Gemma. Results are checkpointed after every case; a stopped run can be resumed. Start with a fresh output directory if you change generation settings.


The default is 10 cases in a separate diagnostic output directory, preserving the original run. GPU inference uses BF16 when supported and FP32 otherwise (including T4); FP16 is avoided. The prompt explicitly supplies a fixed JSON schema, so this is a revised baseline protocol. Use this same protocol for later comparisons. Diagnostics save raw token IDs, decoding with special tokens visible, token counts, first-token logit finiteness, and output-budget hits. Empty responses stop the run and are retried on resume. The first next-token forward pass adds diagnostic overhead.


In [8]:
# 1.1 Data Observation: Look at a sample case from the active selection
sample_case = selection["cases"][0]

print("=" * 80)
print(f"OBSERVING DATA SAMPLE: Index {sample_case['index']}")
print("=" * 80)
print("\n[1. SYSTEM PROMPT / INSTRUCTION]")
print(sample_case["system"])

print("\n" + "-"*40)
print("[2. USER INPUT / CLINICAL NOTE - TRUNCATED]")
# Show first 1000 characters of the input clinical note
print(sample_case["note"][:1000] + "...\n[Truncated for readability]")

print("\n" + "-"*40)
print("[3. EXPECTED ASSISTANT RESPONSE / REFERENCE SUMMARY]")
try:
    # Format reference summary nicely if it's JSON
    formatted_ref = json.dumps(json.loads(sample_case["reference"]), indent=2)
    print(formatted_ref)
except Exception:
    print(sample_case["reference"])
print("=" * 80)

OBSERVING DATA SAMPLE: Index 2619

[1. SYSTEM PROMPT / INSTRUCTION]
You are a clinical documentation specialist. Summarize the clinical note into a structured SOAP format.

----------------------------------------
[2. USER INPUT / CLINICAL NOTE - TRUNCATED]
This case was reported according to the Case REport (CARE) statement and checklist [], and written informed consent was obtained from the patient to publication of this case report.
A 29-year-old female patient, nonsmoker, who reported not consuming alcoholic beverages, not taking any long-term medication, and having no clinically relevant family history information, complained of pain on touching the maxillary left central incisor (tooth 11). Her dental history included injury to the tooth in question due to a fall from standing height during childhood, and the tooth began to be dislodged over the years. In a specialized dental care clinic, she received endodontic treatment that did not achieve the expected success.
Clinical examin

In [9]:
# Fixed output schema: no reference answers are included in the prompt.
JSON_SCHEMA = {
    "visit motivation": "None",
    "admission": [{"reason": "None", "date": "None", "duration": "None", "care center details": "None"}],
    "patient information": {key: "None" for key in (
        "age", "sex", "ethnicity", "weight", "height", "family medical history",
        "recent travels", "socio economic context", "occupation")},
    "patient medical history": {key: "None" for key in (
        "physiological context", "psychological context", "vaccination history", "allergies",
        "exercise frequency", "nutrition", "sexual history", "alcohol consumption", "drug usage", "smoking status")},
    "surgeries": [{key: "None" for key in ("reason", "Type", "time", "outcome", "details")}],
    "symptoms": [{key: "None" for key in (
        "name of symptom", "intensity of symptom", "location", "time", "temporalisation",
        "behaviours affecting the symptom", "details")}],
    "medical examinations": [{key: "None" for key in ("name", "result", "details")}],
    "diagnosis tests": [{key: "None" for key in ("test", "severity", "result", "condition", "time", "details")}],
    "treatments": [{key: "None" for key in (
        "name", "related condition", "dosage", "time", "frequency", "duration",
        "reason for taking", "reaction to treatment", "details")}],
    "discharge": {key: "None" for key in ("reason", "referral", "follow up", "discharge summary")},
}
FORMAT_INSTRUCTION = (
    "Return only one valid JSON object, without markdown or commentary. "
    "Use exactly the keys and nesting in the following schema. "
    "Use strings for values, the string 'None' for missing information, and repeat array "
    "entries when needed. Do not invent facts. Schema:\n"
    + json.dumps(JSON_SCHEMA, ensure_ascii=False)
)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
print("Inference device:", device, "dtype:", dtype)
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Set HF_TOKEN after accepting the Gemma license, then rerun this cell.")
model_revision = HfApi().model_info(MODEL_ID, token=os.environ["HF_TOKEN"]).sha
config = {"model": MODEL_ID, "model_revision": model_revision,
          "dataset_revision": selection["revision"], "seed": SEED,
          "max_new_tokens": MAX_NEW_TOKENS, "do_sample": False,
          "transformers": importlib.metadata.version("transformers"),
          "torch": torch.__version__, "dtype": str(dtype), "device": device,
          "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
          "prompt_version": 2, "format_instruction": FORMAT_INSTRUCTION,
          "selection_sha256": hashlib.sha256(selection_path.read_bytes()).hexdigest()}
config_path = OUTPUT_DIR / "generation_config.json"
if config_path.exists():
    assert json.loads(config_path.read_text(encoding="utf-8")) == config, "Config changed; use a new output directory."
else:
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision, token=os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=model_revision, token=os.environ["HF_TOKEN"],
    torch_dtype=dtype,
).to(device).eval()
outputs_path = OUTPUT_DIR / "generations.json"
outputs = json.loads(outputs_path.read_text(encoding="utf-8")) if outputs_path.exists() else {}
diagnostics_path = OUTPUT_DIR / "generation_diagnostics.json"
diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8")) if diagnostics_path.exists() else {}
for case in selection["cases"]:
    key = str(case["index"])
    if outputs.get(key, "").strip():
        continue
    prompt = case["system"] + "\n\n" + FORMAT_INSTRUCTION + "\n\nClinical note:\n" + case["note"]
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    context_limit = model.config.max_position_embeddings
    prompt_tokens = inputs.input_ids.shape[1]
    if prompt_tokens + MAX_NEW_TOKENS > context_limit:
        raise ValueError(f"Row {key}: prompt ({prompt_tokens}) plus output budget exceeds context ({context_limit}).")
    with torch.inference_mode():
        # Check the first next-token logits before starting a full decode.
        logits = model(**inputs).logits[:, -1, :]
        finite_logits = bool(torch.isfinite(logits).all().item())
        del logits
        if not finite_logits:
            diagnostics[key] = {"prompt_tokens": prompt_tokens, "finite_first_token_logits": False}
            diagnostics_path.write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")
            raise RuntimeError(f"Row {key}: non-finite logits with {dtype}; inspect diagnostics and try FP32.")
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_ids = generated[0, prompt_tokens:].tolist()
    response = tokenizer.decode(new_ids, skip_special_tokens=True)
    diagnostics[key] = {
        "prompt_tokens": prompt_tokens, "generated_tokens": len(new_ids),
        "token_ids": new_ids, "raw_decoded": tokenizer.decode(new_ids, skip_special_tokens=False),
        "finite_first_token_logits": finite_logits,
        "hit_token_limit": len(new_ids) >= MAX_NEW_TOKENS, "empty_output": not response.strip(),
    }
    diagnostics_path.write_text(json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Row {key}: {len(new_ids)} tokens; preview={response[:160]!r}")
    if not response.strip():
        raise RuntimeError(f"Row {key}: empty output. Inspect {diagnostics_path}; this case will be retried on rerun.")
    outputs[key] = response
    outputs_path.write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Generated {len(outputs)}/{N_CASES}: row {key}")
print("Saved:", outputs_path)


Inference device: cuda dtype: torch.bfloat16


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

... (21 outputs left out)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Row 2771: 2048 tokens; preview='```json\n{\n  "visit motivation": "No specific motivation provided.",\n  "admission": [\n    {\n      "reason": "None",\n      "date": "None",\n      "duration": "None'
Generated 9/10: row 2771


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Row 2233: 2048 tokens; preview='```json\n{\n  "visit motivation": "No specific motivation provided.",\n  "admission": [\n    {\n      "reason": "No specific reason provided.",\n      "date": "None",'
Generated 10/10: row 2233
Saved: baseline_evaluation_diagnostic_10/generations.json


## 3. Local JSON structure and word overlap (no judge)

These checks run locally after generation. The parser validates that the generated text is valid JSON and recursively compares its keys against the keys present in the reference JSON summary. Missing and extra keys are flagged.

Reference word recall is the fraction of unique reference words also found in the generated summary; precision is the fraction of unique generated words found in the reference. These are simple case-insensitive word overlaps. No judge, API key, or external call is needed for this step.

In [10]:
import re
import json

def extract_json(text):
    """Attempt to extract a JSON block if wrapped in markdown formatting."""
    text = (text or "").strip()
    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()

def get_all_keys(d, prefix=''):
    """Recursively extract all keys from a dictionary to check schema."""
    keys = set()
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{prefix}.{k}" if prefix else k
            keys.add(full_key)
            keys.update(get_all_keys(v, full_key))
    elif isinstance(d, list):
        for item in d:
            keys.update(get_all_keys(item, prefix))
    return keys

def check_json_structure(generated, reference):
    """Quick fail on non-JSON, and verify key structure against reference."""
    gen_text = extract_json(generated)
    ref_text = extract_json(reference)

    try:
        ref_json = json.loads(ref_text)
        ref_keys = get_all_keys(ref_json)
    except json.JSONDecodeError as exc:
        ref_keys = set()
        reference_error = str(exc)
    else:
        reference_error = None

    result = {
        "is_valid_json": False,
        "reference_is_valid_json": reference_error is None,
        "reference_json_error": reference_error,
        "generated_json_error": None,
        "missing_keys": list(ref_keys),
        "extra_keys": [],
        "exact_key_match": False
    }

    try:
        gen_json = json.loads(gen_text)
        result["is_valid_json"] = True
        gen_keys = get_all_keys(gen_json)

        result["missing_keys"] = sorted(list(ref_keys - gen_keys))
        result["extra_keys"] = sorted(list(gen_keys - ref_keys))
        result["exact_key_match"] = reference_error is None and (ref_keys == gen_keys)
    except json.JSONDecodeError as exc:
        result["generated_json_error"] = str(exc)

    return result

def word_overlap(generated, reference):
    words = lambda text: set(re.findall(r"\b\w+\b", text.casefold()))
    actual, expected = words(generated or ""), words(reference or "")
    common = len(actual & expected)
    return {"reference_word_recall": common / len(expected) if expected else None,
            "generated_word_precision": common / len(actual) if actual else None}

# Small examples make the parser's behavior visible without loading a model.
ref_mock = '{"visit": "pain", "vitals": {"hr": 80}}'
gen_mock = '{"visit": "pain", "vitals": {"bp": 120}}'
check = check_json_structure(gen_mock, ref_mock)
assert check["is_valid_json"]
assert "vitals.hr" in check["missing_keys"]
assert "vitals.bp" in check["extra_keys"]
print("Parser self-checks passed.")
assert not check_json_structure("", ref_mock)["is_valid_json"]
assert not check_json_structure("{}", "invalid")["exact_key_match"]
assert check_json_structure("{}", "invalid")["reference_json_error"]
assert check_json_structure('```json\n' + ref_mock + '\n```', ref_mock)["exact_key_match"]


Parser self-checks passed.


In [11]:
outputs_path = OUTPUT_DIR / "generations.json"
outputs = json.loads(outputs_path.read_text(encoding="utf-8")) if outputs_path.exists() else {}
local_results = []
for case in selection["cases"]:
    key = str(case["index"])
    if key not in outputs:
        continue  # Missing generations are not counted as empty model outputs.

    local_results.append({
        "index": case["index"],
        "json_structure": check_json_structure(outputs[key], case["reference"]),
        "word_overlap": word_overlap(outputs[key], case["reference"])
    })

local_path = OUTPUT_DIR / "local_checks.json"
local_path.write_text(json.dumps(local_results, ensure_ascii=False, indent=2), encoding="utf-8")

local_summary = {
    "generated_cases_checked": len(local_results),
    "empty_output_cases": sum(not outputs[str(row["index"])].strip() for row in local_results),
    "invalid_reference_cases": sum(not row["json_structure"]["reference_is_valid_json"] for row in local_results),
    "valid_json_cases": sum(1 for row in local_results if row["json_structure"]["is_valid_json"]),
    "exact_key_match_cases": sum(1 for row in local_results if row["json_structure"]["exact_key_match"])
}

for name in ("reference_word_recall", "generated_word_precision"):
    values = [row["word_overlap"][name] for row in local_results if row["word_overlap"][name] is not None]
    local_summary[name] = {"count": len(values), "mean": sum(values) / len(values) if values else None}

(OUTPUT_DIR / "local_aggregate.json").write_text(json.dumps(local_summary, indent=2), encoding="utf-8")
print(json.dumps(local_summary, indent=2))
print("Per-case details:", local_path)

{
  "generated_cases_checked": 10,
  "empty_output_cases": 0,
  "invalid_reference_cases": 2,
  "valid_json_cases": 5,
  "exact_key_match_cases": 3,
  "reference_word_recall": {
    "count": 10,
    "mean": 0.21135305353507308
  },
  "generated_word_precision": {
    "count": 10,
    "mean": 0.8825427905427905
  }
}
Per-case details: baseline_evaluation_diagnostic_10/local_checks.json


## 4. Judge with DeepEval (optional paid API calls)

Set `RUN_PAID_JUDGING = True` only after reviewing the cost and data transfer above. The Gemini Developer API judge is separate from the baseline model; it uses `GOOGLE_API_KEY`, and does not require an OpenAI key. The notebook does not configure a custom OpenAI-compatible base URL. Other providers require modifying the judge integration; see [DeepEval model integrations](https://deepeval.com/integrations). Each G-Eval metric scores 0–1 and provides a reason: **faithfulness** checks the source note for unsupported or contradicted claims; **coverage** checks whether important source and reference facts appear in the generated summary. References can themselves contain errors, so the source note is authoritative. These scores are noisy and are not proof of medical accuracy. Each completed score is saved immediately; rerun to resume after an interruption.


In [ ]:
RUN_PAID_JUDGING = False  # Change to True to authorize API calls for this run.
if not RUN_PAID_JUDGING:
    print("Judging skipped. Review costs and set RUN_PAID_JUDGING = True to evaluate.")
else:
    if not os.environ.get("GOOGLE_API_KEY"):
        raise RuntimeError("Set GOOGLE_API_KEY in Colab Secrets or the environment.")
    from deepeval.metrics import GEval
    from deepeval.models import GeminiModel
    from deepeval.test_case import LLMTestCase, SingleTurnParams

    assert all(outputs.get(str(c["index"]), "").strip() for c in selection["cases"]), "Complete nonempty generation first."
    judge_config = {"provider": "Gemini Developer API", "model": JUDGE_MODEL, "deepeval": importlib.metadata.version("deepeval"),
                    "selection_sha256": hashlib.sha256(selection_path.read_bytes()).hexdigest(),
                    "generations_sha256": hashlib.sha256(outputs_path.read_bytes()).hexdigest()}
    judge_config_path = OUTPUT_DIR / "judge_config.json"
    if judge_config_path.exists():
        assert json.loads(judge_config_path.read_text(encoding="utf-8")) == judge_config, "Judge or inputs changed; use a new output directory."
    else:
        judge_config_path.write_text(json.dumps(judge_config, indent=2), encoding="utf-8")

    judge = GeminiModel(model=JUDGE_MODEL, api_key=os.environ["GOOGLE_API_KEY"], use_vertexai=False, temperature=0)

    metrics = {
        "faithfulness": GEval(name="Factual faithfulness", model=judge, async_mode=False,
            evaluation_steps=["Analyze the actual output (which is formatted as JSON) and extract its clinical claims.",
                              "Compare every claim in the actual output JSON with the source note in input.",
                              "Penalize unsupported, invented, or contradicted clinical facts, especially diagnoses, medications and quantities."],
            evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT]),
        "coverage": GEval(name="Clinical coverage", model=judge, async_mode=False,
            evaluation_steps=["Identify important clinical facts in the source note in input, using expected output JSON as a guide.",
                              "Check whether actual output JSON preserves these facts; penalize important omissions. The source note is authoritative if the reference conflicts."],
            evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT]),
    }

    scores_path = OUTPUT_DIR / "scores.json"
    scores = json.loads(scores_path.read_text(encoding="utf-8")) if scores_path.exists() else {}
    for case in selection["cases"]:
        key = str(case["index"])
        scores.setdefault(key, {})
        test_case = LLMTestCase(input=case["note"], actual_output=outputs[key], expected_output=case["reference"])
        for name, metric in metrics.items():
            if name in scores[key]:
                continue
            metric.measure(test_case)
            scores[key][name] = {"score": metric.score, "reason": metric.reason}
            scores_path.write_text(json.dumps(scores, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"Row {key}, {name}: {metric.score:.3f}")
    print("Saved:", scores_path)

## 5. Inspect and export results

This cell reports available scores and their counts, including partial runs. `per_case.csv` includes the note, reference, output, scores and judge explanations. Compare later models on the same saved selection, prompt, generation settings and judge configuration; the judge may still vary between runs.


In [ ]:
outputs = json.loads((OUTPUT_DIR / "generations.json").read_text(encoding="utf-8")) if (OUTPUT_DIR / "generations.json").exists() else {}
scores = json.loads((OUTPUT_DIR / "scores.json").read_text(encoding="utf-8")) if (OUTPUT_DIR / "scores.json").exists() else {}
rows = []
for case in selection["cases"]:
    key = str(case["index"])
    row = {"index": case["index"], "sha256": case["sha256"], "note": case["note"],
           "reference": case["reference"], "generated_soap": outputs.get(key)}
    for name in ("faithfulness", "coverage"):
        result = scores.get(key, {}).get(name, {})
        row[name + "_score"] = result.get("score")
        row[name + "_reason"] = result.get("reason")
    rows.append(row)
results = pd.DataFrame(rows)
results.to_csv(OUTPUT_DIR / "per_case.csv", index=False)
aggregate = {"selected_cases": len(results), "generated_cases": sum(bool(outputs.get(str(c["index"]), "").strip()) for c in selection["cases"])}
aggregate["empty_cases"] = sum(str(c["index"]) in outputs and not outputs[str(c["index"])].strip() for c in selection["cases"])
for name in ("faithfulness", "coverage"):
    values = pd.to_numeric(results[name + "_score"], errors="coerce").dropna()
    aggregate[name] = {"count": int(values.count()), "mean": float(values.mean()) if len(values) else None}
(OUTPUT_DIR / "aggregate.json").write_text(json.dumps(aggregate, indent=2), encoding="utf-8")
print(json.dumps(aggregate, indent=2))
display(results[["index", "generated_soap", "faithfulness_score", "coverage_score"]].head())


In [12]:
import shutil

shutil.make_archive(
    "/content/baseline_evaluation_diagnostic_10",
    "zip",
    "/content",
    "baseline_evaluation_diagnostic_10",
)

'/content/baseline_evaluation_diagnostic_10.zip'